In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)


In [ ]:
df = pd.read_csv('../../data/courses.csv')

# =========================================================
# 불필요한 컬럼 제거
df.drop(columns='roles', inplace=True)

# havardX 에서는 거의 90퍼센트가 결측치다.
# MIT 의 경우 50.4퍼센트에 육박. 그 중에서도 viewed = 1 인데도 결측이 36% 존재. => 동영상을 봤는데 기록이 없다. 0으로 대체하면 실제로 영상을 안 본 학생이라고 판단하는 셈. 그런데 그대로 두면 전체 결측치가 73% 정도로 매우 많기 때문에 분석에 활용하기 어렵다고 판단.
# => nplay_video 컬럼은 분석에는 제외하고 필요할 때 참고한다.
df.drop(columns='nplay_video', inplace=True)
# =========================================================

# 컬럼 전체 소문자 변경
df.columns = df.columns.str.lower()

# 컬럼명 변경
df.rename(columns={'userid_di': 'userid', 'final_cc_cname_di': 'country', 'loe_di': 'loe', 'start_time_di': 'start_time', 'last_event_di': 'last_event'}, inplace=True)

# str => date 타입 변경
df['start_time'] = pd.to_datetime(df['start_time'])
df['last_event'] = pd.to_datetime(df['last_event'])

# str => float 타입 변경
df['grade'] = pd.to_numeric(df['grade'], errors='coerce')

# float => int 타입 변환
df[['nevents', 'ndays_act', 'nchapters', 'incomplete_flag', 'yob']] = df[['nevents', 'ndays_act', 'nchapters', 'incomplete_flag', 'yob']].astype('Int64')

# 데이터 소문자 변경
df['country'] = df['country'].str.lower()
df['userid'] = df['userid'].str.lower()
df['loe'] = df['loe'].str.lower()
df['course_id'] = df['course_id'].str.lower()
df['gender'] = df['gender'].str.lower()

# 결측치 대체
df['grade'] = df['grade'].fillna(0)
df['loe'] = df['loe'].fillna('unknown')
df['gender'] = df['gender'].fillna('unknown')

# loe 그룹별 yob 중앙값
yob_median_by_loe = df.groupby('loe')['yob'].median().reset_index()
yob_median_by_loe.rename(columns={'yob': 'yob_median'}, inplace=True)

# yob 결측에 그룹별로 채워넣기
df = df.merge(yob_median_by_loe, on='loe', how='left')
df.loc[df['yob'].isna(), 'yob'] = df.loc[df['yob'].isna(), 'yob_median']
df['yob'] = df['yob'].astype('Int64')
df.drop(columns='yob_median', inplace=True)

# gender 'o' -> 'unknown'으로 변경
df.loc[df['gender'] == 'o', 'gender'] = 'unknown'

# grade = 0 & certified = 1 인 값 확인 및 제거
target_users = df.loc[(df['grade'] == 0) & (df['certified'] == 1), 'userid']
df = df[~df['userid'].isin(target_users)]

# viewed = 0 인데 explored = 1인 행 제거
target_users = df.loc[(df['viewed'] == 0) & (df['explored'] == 1), 'userid']
df = df[~df['userid'].isin(target_users)]

# duration 파생컬럼 생성
df['duration'] = (df['last_event'] - df['start_time']).dt.days

# 13세 미만 사용자 제거
df['age'] = pd.to_datetime(df['start_time']).dt.year - df['yob']
df = df[df['age'] >= 13]

# 파생컬럼 생성
# course_id 소문자 변경 후 기관/강의코드/강의년도/강의학기 생성
df[['institution', 'course_code', 'year_term']] = df['course_id'].str.split('/', expand=True)
df[['course_year', 'course_term']] = df['year_term'].str.split('_', expand=True)
df['course_term'] = df['course_term'].fillna('unknown')
df.drop(columns='year_term', inplace=True)

# course_id 소문자 기준으로 카테고리 매핑
category_map = {
    'harvardx/cb22x/2013_spring': 'The Ancient Greek Hero',
    'harvardx/cs50x/2012': 'Introduction to Computer Science',
    'harvardx/er22x/2013_spring': 'Justice',
    'harvardx/ph207x/2012_fall': 'Health in Numbers: Quantitative Methods in Clinical and Public Health Research',
    'harvardx/ph278x/2013_spring': 'Human Health and Global Environmental Change',
    'mitx/14.73x/2013_spring': 'The Challenges of Global Poverty',
    'mitx/2.01x/2013_spring': 'Elements of Structures',
    'mitx/3.091x/2012_fall': 'Introduction to Solid State Chemistry',
    'mitx/3.091x/2013_spring': 'Introduction to Solid State Chemistry',
    'mitx/6.002x/2012_fall': 'Circuits and Electronics',
    'mitx/6.002x/2013_spring': 'Circuits and Electronics',
    'mitx/6.00x/2012_fall': 'Introduction to Computer Science and Programming',
    'mitx/6.00x/2013_spring': 'Introduction to Computer Science and Programming',
    'mitx/7.00x/2013_spring': 'Introduction to Biology: The Secret of Life',
    'mitx/8.02x/2013_spring': 'Electricity and Magnetism',
    'mitx/8.mrev/2013_summer': 'Mechanics ReView',
}

df['course_category'] = df['course_id'].map(category_map)
df.info()

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/courses.csv")

# =========================================================
# [0] 기본 전처리 (네 코드 기반, 몇 군데만 안전하게 정리)
# =========================================================

# 불필요 컬럼 제거
if "roles" in df.columns:
    df.drop(columns="roles", inplace=True)

# nplay_video는 분석 제외
if "nplay_video" in df.columns:
    df.drop(columns="nplay_video", inplace=True)

# 컬럼 소문자
df.columns = df.columns.str.lower()

# 컬럼명 변경
df.rename(
    columns={
        "userid_di": "userid",
        "final_cc_cname_di": "country",
        "loe_di": "loe",
        "start_time_di": "start_time",
        "last_event_di": "last_event",
    },
    inplace=True,
)

# 날짜 변환
df["start_time"] = pd.to_datetime(df["start_time"], errors="coerce")
df["last_event"] = pd.to_datetime(df["last_event"], errors="coerce")

# grade: float 유지 (정수로 바꾸지 않음)
df["grade"] = pd.to_numeric(df["grade"], errors="coerce")

# 문자열 소문자
for c in ["country", "userid", "loe", "course_id", "gender"]:
    if c in df.columns:
        df[c] = df[c].astype("string").str.lower()

# 결측치 기본 대체
df["grade"] = df["grade"].fillna(0)
df["loe"] = df["loe"].fillna("unknown")
df["gender"] = df["gender"].fillna("unknown")

# yob: loe 그룹 중앙값으로 채우기
df["yob"] = pd.to_numeric(df["yob"], errors="coerce")
yob_median_by_loe = df.groupby("loe")["yob"].median()
df.loc[df["yob"].isna(), "yob"] = df.loc[df["yob"].isna(), "loe"].map(yob_median_by_loe)
df["yob"] = df["yob"].round(0).astype("Int64")

# gender 'o' -> unknown
df.loc[df["gender"] == "o", "gender"] = "unknown"

# 이상 케이스 제거
# grade=0 & certified=1 사용자 제거
df["certified"] = pd.to_numeric(df["certified"], errors="coerce").fillna(0).astype(int)
df["viewed"] = pd.to_numeric(df["viewed"], errors="coerce").fillna(0).astype(int)
df["explored"] = pd.to_numeric(df["explored"], errors="coerce").fillna(0).astype(int)

bad_users = df.loc[(df["grade"] == 0) & (df["certified"] == 1), "userid"]
df = df[~df["userid"].isin(bad_users)].copy()

# viewed=0 & explored=1 사용자 제거
bad_users = df.loc[(df["viewed"] == 0) & (df["explored"] == 1), "userid"]
df = df[~df["userid"].isin(bad_users)].copy()

# duration, age
df["age"] = df["start_time"].dt.year - df["yob"]
df = df[df["age"] >= 13].copy()

# course_id 분해 파생
df[["institution", "course_code", "year_term"]] = df["course_id"].str.split("/", expand=True)
df[["course_year", "course_term"]] = df["year_term"].str.split("_", expand=True)
df["course_term"] = df["course_term"].fillna("unknown")
df.drop(columns="year_term", inplace=True)

# course_category 매핑
category_map = {
    'harvardx/cb22x/2013_spring': 'Humanities, History, Design, Religion, and Education',
    'harvardx/cs50x/2012': 'Computer Science',
    'harvardx/er22x/2013_spring': 'Humanities, History, Design, Religion, and Education',
    'harvardx/ph207x/2012_fall': 'Government, Health, and Social Science',
    'harvardx/ph278x/2013_spring': 'Government, Health, and Social Science',
    'mitx/14.73x/2013_spring': 'Government, Health, and Social Science',
    'mitx/2.01x/2013_spring': 'Science, Technology, Engineering, and Mathematics',
    'mitx/3.091x/2012_fall': 'Science, Technology, Engineering, and Mathematics',
    'mitx/3.091x/2013_spring': 'Science, Technology, Engineering, and Mathematics',
    'mitx/6.002x/2012_fall': 'Science, Technology, Engineering, and Mathematics',
    'mitx/6.002x/2013_spring': 'Science, Technology, Engineering, and Mathematics',
    'mitx/6.00x/2012_fall': 'Computer Science',
    'mitx/6.00x/2013_spring': 'Computer Science',
    'mitx/7.00x/2013_spring': 'Science, Technology, Engineering, and Mathematics',
    'mitx/8.02x/2013_spring': 'Science, Technology, Engineering, and Mathematics',
    'mitx/8.mrev/2013_summer': 'Science, Technology, Engineering, and Mathematics',
}
df["course_category"] = df["course_id"].map(category_map)

# =========================================================
# [1] viewed=0/1 결측 처리 파이프라인 합치기 (에러 안 나게 dtype 처리 포함)
# =========================================================

stage_cols = ["viewed", "explored", "certified"]
n_cols = ["nevents", "ndays_act", "nchapters"]
group_cols = ["course_id", "viewed", "explored", "certified"]

# n_cols는 중앙값(median) 채우면 소수 가능성이 생김 -> 일단 float로 통일(중요!)
df[n_cols] = df[n_cols].apply(pd.to_numeric, errors="coerce").astype("float64")

# [A] viewed=0 처리
# A-1-1) viewed=0 & nchapters 존재(notna) -> 삭제
mask_drop = (df["viewed"] == 0) & (df["nchapters"].notna())
drop_count = int(mask_drop.sum())
df = df.loc[~mask_drop].copy()

# A-1-2) viewed=0 & nchapters 결측 -> nchapters=0
mask_fill_nchapters0 = (df["viewed"] == 0) & (df["nchapters"].isna())
df.loc[mask_fill_nchapters0, "nchapters"] = 0

# A-2) viewed=0 & n_시리즈 전부 결측 -> 비활동자 + 0채움
df["inactive_flag"] = 0
mask_inactive = (df["viewed"] == 0) & df[n_cols].isna().all(axis=1)
df.loc[mask_inactive, "inactive_flag"] = 1
df.loc[mask_inactive, n_cols] = 0

# A-3) 사이트 구경 집단 -> 그룹 중앙값으로 결측 채움
mask_browse = (df["viewed"] == 0) & (df["inactive_flag"] == 0)
df.loc[mask_browse, n_cols] = df.loc[mask_browse, n_cols].fillna(
    df.loc[mask_browse].groupby(group_cols)[n_cols].transform("median")
)

# [B] viewed=1 처리 -> 그룹 중앙값으로 결측 채움
mask_viewed1 = (df["viewed"] == 1)
df.loc[mask_viewed1, n_cols] = df.loc[mask_viewed1, n_cols].fillna(
    df.loc[mask_viewed1].groupby(group_cols)[n_cols].transform("median")
)

# (선택) n_cols를 정수로 유지하고 싶으면 여기서 반올림 후 Int64로
# - 행동지표면 보통 정수 의미가 강해서 round 추천
df[n_cols] = df[n_cols].round(0).astype("Int64")

# =========================================================
# [2] duration 보정(결측 -> ndays_act, 음수 -> ndays_act)
# =========================================================
df["duration"] = (df["last_event"] - df["start_time"]).dt.days
df["duration"] = pd.to_numeric(df["duration"]+1, errors="coerce")
df["ndays_act"] = pd.to_numeric(df["ndays_act"], errors="coerce")

mask_duration_na = df["duration"].isna()
df.loc[mask_duration_na, "duration"] = df.loc[mask_duration_na, "ndays_act"]

mask_duration_neg = df["duration"] < df['ndays_act']
df.loc[mask_duration_neg, "duration"] = df.loc[mask_duration_neg, "ndays_act"]

# df["duration"] = pd.to_numeric(df["duration"], errors="coerce")
df['duration'] = df['duration'].astype('Int64')

# =========================================================
# [3] grade 보정: 1.01 -> 1.0 (float 유지)
# =========================================================
df.loc[df["grade"].eq(1.01), "grade"] = 1.0
# (대안) 1 초과는 전부 1로 cap 하고 싶으면:
# df["grade"] = df["grade"].clip(upper=1.0)

# =========================================================
# [4] student_type 생성(원하면)
# =========================================================
conditions = [
    (df["viewed"] == 0) & (df["explored"] == 0) & (df["certified"] == 0),
    (df["viewed"] == 1) & (df["explored"] == 0) & (df["certified"] == 0),
    (df["viewed"] == 1) & (df["explored"] == 1) & (df["certified"] == 0),
    (df["viewed"] == 1) & (df["certified"] == 1),  # completer: explored 0/1 모두 포함
]
choices = ["viewered", "viewer", "explorer", "completer"]
df["student_type"] = np.select(conditions, choices, default="unknown")

# 2) (선택) 혹시 아직 결측이 남아있으면 채우고 정수화해야 함
#    - 이미 ndays_act로 채웠다면 이 줄은 필요 없을 수 있어
# df2["duration"] = df2["duration"].fillna(df2["ndays_act"])

# =========================================================
# [5] 마무리/저장
# =========================================================
df2 = df.copy()

# 필요 없으면 제거
for c in ["index", "incomplete_flag"]:
    if c in df2.columns:
        df2.drop(columns=c, inplace=True)

# print("\n[삭제된 행 수(viewed=0 & nchapters notna)]", drop_count)
# print("[비활동자 수(inactive_flag=1)]", int(df2["inactive_flag"].sum()))
# print("\n[n_cols 결측률 - 전체]")
# print(df2[n_cols].isna().mean())

df2.to_csv("../../data/preprocessed.csv", index=False)